# 04 — Adaptive local contrast Development Evaluation

Evaluate this run's selected detector on its own policy's development (test) split with native FP32 Ultralytics validation, saving per-image matches for dark/bright and Apply/Skip cohorts.

**Inputs:** runs/v3-YYYY-MM-DD/train/weights/best.pt and the run's prepared data.

**Outputs:** runs/v3-YYYY-MM-DD/eval/{metrics.json,paired_stats.npz,plots} only when ALLOW_INFERENCE=True.

**Prerequisites:** Completed 02 and 03. Evaluation never imports or invokes training.

**Outline**

- [1. Configuration](#section-1)
- [2. Policy–Detector Pair](#section-2)
- [3. Evaluation Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

RUN_DATE=None uses the newest run. Saved metrics are reused when inputs match.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from vehicle_detection.common import require_paths
from vehicle_detection.evaluation import evaluate_experiment
from vehicle_detection.experiment_data import load_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
EXPERIMENT = "v3"
require_paths(
    config=ROOT / f"notebooks/experiments/{EXPERIMENT}/{EXPERIMENT}.yaml",
    shared=ROOT / "configs/data.yaml",
)
config = load_experiment(ROOT, EXPERIMENT)

RUN_DATE = None
ALLOW_INFERENCE = False
display(pd.Series(config["evaluation"], name="evaluation"))

<a id="section-2"></a>

## 2. Policy–Detector Pair

Only development is evaluated; the final holdout is never read.

In [ ]:
try:
    result = evaluate_experiment(ROOT, EXPERIMENT, RUN_DATE, allow_inference=ALLOW_INFERENCE)
except FileNotFoundError as error:
    result = None
    print(error)
if result:
    display(pd.DataFrame([result["overall"]]))
    display(pd.DataFrame(result["classes"]).set_index("Class"))

<a id="section-3"></a>

## 3. Evaluation Summary

Cohort AP keeps every GT of the selected images; dark means raw brightness < 0.12.

In [ ]:
if result:
    display(
        pd.DataFrame(
            {
                k: {"images": v["images"], "mAP50_95": v["mAP50_95"]}
                for k, v in result["cohorts"].items()
            }
        ).T
    )
    print("Run:", result["inputs"]["run"], "| speed (ms):", result["speed_ms"])
print("Final holdout stays sealed.")